In [1]:
import keras
import pandas as pd
import numpy as np
import math
import matplotlib as plt
import os
import tensorflow as tf

2026-07-21 17:14:06.976292: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-21 17:14:07.458132: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-21 17:14:10.421989: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
@keras.utils.register_keras_serializable(package="custom_layers")
class Genetic_Algorithm(keras.layers.Layer):
    def __init__(self, name=None, kernel_initializer=None, **kwargs):
        print("CLK__init__----")
        super().__init__(**kwargs)

        self.population_size = 20
        self.population = None
        self.x_train, self.y_train = Get_Data("train.csv", is_train=True)

    def build(self, input_shape):
        print("CLKbuild----")
        print(input_shape)

        num_features = input_shape[-1]

        self.population = tf.Variable(
            tf.random.normal((self.population_size, num_features)),
            trainable=False,
            dtype=tf.float32,
        )

        super().build(input_shape)

    def call(self, inputs, model):
        rinputs, kept = Reduce(self.x_train)
        ry_train = tf.gather(self.y_train, kept)
        losses = Evaluate_Genes(self.population, model, rinputs, ry_train)
        new_population, best_gene = Evolve(self.population, losses)
        self.population.assign(new_population)
        return inputs * best_gene
    
# bce = tf.keras.losses.BinaryCrossentropy()
# loss = bce(true_labels, predictions).numpy()

@keras.utils.register_keras_serializable(package="custom_layers")
class CMGAL2(keras.Model):
    def __init__(self, units, **kwargs):
        super(CMGAL2, self).__init__(**kwargs)
        print("CM__init__----")
        self.GA = Genetic_Algorithm(name="GA")
        self.layer1 = keras.layers.Dense(units=2048,activation=keras.activations.leaky_relu)
        self.layer2 = keras.layers.Dense(units=1024, activation=keras.activations.leaky_relu)
        self.layer3 = keras.layers.Dense(units=512, activation=keras.activations.leaky_relu)
        self.final = keras.layers.Dense(units=1, activation=keras.activations.sigmoid)
        pass

    def build(self, input_shape):
        print("CMbuild----")
        #super.build()
        pass

    def call(self, inputs):
        print("CMcall----")
        x1 = self.layer1(inputs)
        x2 = self.GA(x1, model=[self.layer1, "GA", self.layer2, self.layer3, self.final])
        x3 = self.layer2(x2)
        x4 = self.layer3(x3)
        x5 = self.final(x4)
        return x5

@keras.utils.register_keras_serializable(package="custom_layers")
class CMGAL3(keras.Model):
    def __init__(self, units, **kwargs):
        super(CMGAL3, self).__init__(**kwargs)
        print("CM__init__----")
        self.GA = Genetic_Algorithm(name="GA")
        self.layer1 = keras.layers.Dense(units=2048,activation=keras.activations.leaky_relu)
        self.layer2 = keras.layers.Dense(units=1024, activation=keras.activations.leaky_relu)
        self.layer3 = keras.layers.Dense(units=512, activation=keras.activations.leaky_relu)
        self.final = keras.layers.Dense(units=1, activation=keras.activations.sigmoid)
        pass

    def build(self, input_shape):
        print("CMbuild----")
        #super.build()
        pass

    def call(self, inputs):
        print("CMcall----")
        x1 = self.layer1(inputs)
        x2 = self.layer2(x1)
        x3 = self.GA(x2, model=[self.layer1, self.layer2, "GA", self.layer3, self.final])
        x4 = self.layer3(x3)
        x5 = self.final(x4)
        return x5
    
@keras.utils.register_keras_serializable(package="custom_layers")
class CMGAL4(keras.Model):
    def __init__(self, units, **kwargs):
        super(CMGAL4, self).__init__(**kwargs)
        print("CM__init__----")
        self.GA = Genetic_Algorithm(name="GA")
        self.layer1 = keras.layers.Dense(units=2048,activation=keras.activations.leaky_relu)
        self.layer2 = keras.layers.Dense(units=1024, activation=keras.activations.leaky_relu)
        self.layer3 = keras.layers.Dense(units=512, activation=keras.activations.leaky_relu)
        self.final = keras.layers.Dense(units=1, activation=keras.activations.sigmoid)
        pass

    def build(self, input_shape):
        print("CMbuild----")
        #super.build()
        pass

    def call(self, inputs):
        print("CMcall----")
        x1 = self.layer1(inputs)
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.GA(x2, model=[self.layer1, self.layer2, self.layer3, "GA", self.final])
        x5 = self.final(x4)
        return x5
    
@keras.utils.register_keras_serializable(package="custom_layers")
class CustomModelWithoutGA(keras.Model):
    def __init__(self, units, **kwargs):
        super(CustomModelWithoutGA, self).__init__(**kwargs)
        print("CM__init__----")

        self.layer1 = keras.layers.Dense(units=2048,activation=keras.activations.leaky_relu)
        self.layer2 = keras.layers.Dense(units=1024, activation=keras.activations.leaky_relu)
        self.layer3 = keras.layers.Dense(units=512, activation=keras.activations.leaky_relu)
        self.final = keras.layers.Dense(units=1, activation=keras.activations.sigmoid)
        pass

    def build(self, input_shape):
        print("CMbuild----")
        #super.build()
        pass

    def call(self, inputs):
        print("CMcall----")
        x1 = self.layer1(inputs)
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.final(x3)
        return x4

def Reduce(arr):
    n = tf.shape(arr)[0]
    keep_n = n - n // 4
    scores = tf.random.uniform([n])
    keep_indices = tf.math.top_k(scores, k=keep_n).indices
    keep_indices = tf.sort(keep_indices)

    return tf.gather(arr, keep_indices), keep_indices

def Get_Data(file, is_train=True):
    data = pd.read_csv("Titanic/"+file)

    y = None
    Pid = data["PassengerId"].tolist()

    if is_train:
        y = data["Survived"]
        data = data.drop(["Name", "Survived", "Ticket", "Cabin"], axis=1)
    else:
        data = data.drop(["Name", "Ticket", "Cabin"], axis=1)

    """name = data["Name"].tolist()
    for z in range(len(name)):
        name[z] = name[z].split(",")[0]
    for z in range(len(name)):
        name[z] = name.index(name[z]) + 1
    data["Name"] = name"""

    fare = data["Fare"].tolist()
    data["Fare"] = fare

    gender = data["Sex"].tolist()
    for z in range(len(gender)):
        gender[z] = 2 if gender[z] == "male" else 1
    data["Sex"] = gender

    embarked = data["Embarked"].fillna("S").tolist()
    data = data.drop(["Embarked"], axis=1)

    LeftC, LeftQ, LeftS = [], [], []
    for x in embarked:
        if x == "C":
            LeftC.append(1)
            LeftQ.append(0)
            LeftS.append(0)
        elif x == "Q":
            LeftC.append(0)
            LeftQ.append(1)
            LeftS.append(0)
        elif x == "S":
            LeftC.append(0)
            LeftQ.append(0)
            LeftS.append(1)
        else:
            print(x)
            print("LALALALLALALALALLALALALALLALALALALAL")
        

    data.insert(2, "LeftC", LeftC)
    data.insert(2, "LeftQ", LeftQ)
    data.insert(2, "LeftS", LeftS)

    age = data["Age"].tolist()
    avgage = np.nanmean(age)
    age = [avgage if np.isnan(a) else a for a in age]
    data["Age"] = age

    return (data, y) if is_train else (data, Pid)

"""def Evolve(population, fitness):
    pop_size = tf.shape(population)[0]
    elite_size = pop_size // 4

    order = tf.argsort(fitness, direction="DESCENDING")
    elites = tf.gather(population, order)[:elite_size]
    Best_gene = elites[0]

    children = []
    for _ in range(tf.math.multiply(elite_size, 2)):
        i = tf.random.uniform([], 0, elite_size, dtype=tf.int32)
        j = tf.random.uniform([], 0, elite_size, dtype=tf.int32)

        p1 = elites[i]
        p2 = elites[j]

        mask = tf.random.uniform(tf.shape(p1)) < 0.5
        children.append(tf.where(mask, p1, p2))

    children = tf.stack(children)

    mutants = tf.gather(
        elites,
        tf.random.uniform([elite_size], 0, elite_size, dtype=tf.int32)
    )

    mutants += tf.random.uniform(tf.shape(mutants), -0.2, 0.2)

    return tf.concat([elites, children, mutants], axis=0), Best_gene"""

def Evolve(population, fitness):
    pop_size = tf.shape(population)[0]
    num_features = tf.shape(population)[1]
    elite_size = pop_size // 4
    num_children = elite_size * 2

    # --- selection ---
    order = tf.argsort(fitness, direction="DESCENDING")
    elites = tf.gather(population, order[:elite_size])
    Best_gene = elites[0]

    # --- crossover: vectorized, no Python loop ---
    i_idx = tf.random.uniform([num_children], 0, elite_size, dtype=tf.int32)
    j_idx = tf.random.uniform([num_children], 0, elite_size, dtype=tf.int32)

    p1 = tf.gather(elites, i_idx)
    p2 = tf.gather(elites, j_idx)

    mask = tf.random.uniform(tf.shape(p1)) < 0.5
    children = tf.where(mask, p1, p2)

    # --- mutation ---
    mutant_idx = tf.random.uniform([elite_size], 0, elite_size, dtype=tf.int32)
    mutants = tf.gather(elites, mutant_idx)
    mutants += tf.random.uniform(tf.shape(mutants), -0.2, 0.2)

    return tf.concat([elites, children, mutants], axis=0), Best_gene

"""def Evaluate_Genes(self, population, model, inputs, y_train):
    loss = tf.keras.losses.BinaryCrossentropy()
    Eval = []
    for gene in population:
        x = inputs
        for z in model:
            if(z == "GA"):
                x = x * gene
            else:
                x = z(x)
        Eval.append(loss(y_train, x))
    return Eval"""

def Evaluate_Genes(population, model, inputs, y_train):
    loss_fn = tf.keras.losses.BinaryCrossentropy(reduction=tf.keras.losses.Reduction.NONE)
    x = tf.expand_dims(inputs, axis=0)
    genes = tf.expand_dims(population, axis=1)
    for layer in model:
        if layer == "GA":
            x = x * genes
        else:
            x = layer(x)
    if x.shape.ndims == 3 and x.shape[-1] == 1:
        x = tf.squeeze(x, axis=-1)

    y_true = tf.reshape(tf.cast(y_train, x.dtype), [1, -1])
    y_true = tf.broadcast_to(y_true, tf.shape(x))

    Eval = loss_fn(y_true, x)
    return Eval

def Plot_Loss(Data):
    fig, ax = plt.pyplot.subplots()
    ax.yaxis.set_major_locator(plt.ticker.MaxNLocator(10))
    #print(history.history)
    
    ploty = Data[0].history["loss"]
    plotx = np.linspace(0, len(ploty), len(ploty))
    ax.plot(plotx,ploty, label="CMGAL2")

    ploty = Data[1].history["loss"]
    plotx = np.linspace(0, len(ploty), len(ploty))
    ax.plot(plotx,ploty, label="CMGAL3")

    ploty = Data[2].history["loss"]
    plotx = np.linspace(0, len(ploty), len(ploty))
    ax.plot(plotx,ploty, label="CMGAL4")

    ploty = Data[3].history["loss"]
    plotx = np.linspace(0, len(ploty), len(ploty))
    ax.plot(plotx,ploty, label="CustomModelWithoutGA")


    # ax.set_yscale("log")
    # ax.set_xscale("log")
    plt.legend_handlers = plt.pyplot.legend(loc="upper right")
    plt.pyplot.savefig("plot.png")


In [ ]:
x_train, y_train = Get_Data("train.csv", is_train=True)
x_train = tf.convert_to_tensor(x_train, dtype=tf.float32)
y_train = tf.convert_to_tensor(y_train, dtype=tf.float32)
x_test, Pid = Get_Data("test.csv", is_train=False)
x_test = tf.convert_to_tensor(x_test, dtype=tf.float32)
"""print(x_train, "\n")
print(y_train, "\n")
print(x_test)"""
print(x_train)

AttributeError: 'DataFrame' object has no attribute 'dtype'

In [4]:

model2 = CMGAL2(2)
model3 = CMGAL3(2)
model4 = CMGAL4(2)
model5 = CustomModelWithoutGA(2)

model2.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001), 
    loss=keras.losses.BinaryCrossentropy, #           
)
model3.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001), 
    loss=keras.losses.BinaryCrossentropy, #           
)
model4.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001), 
    loss=keras.losses.BinaryCrossentropy, #           
)
model5.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001), 
    loss=keras.losses.BinaryCrossentropy, #           
)

EPOCHS = 50

print("---START---")

history2 = model2.fit(x_train,y_train, epochs=EPOCHS, verbose=1)
history3 = model3.fit(x_train,y_train, epochs=EPOCHS, verbose=1)
history4 = model4.fit(x_train,y_train, epochs=EPOCHS, verbose=1)
history5 = model5.fit(x_train,y_train, epochs=EPOCHS, verbose=1)

Plot_Loss([history2, history3, history4, history5])


CM__init__----
CLK__init__----
CM__init__----
CLK__init__----
CM__init__----
CLK__init__----
CM__init__----
---START---
Epoch 1/50
CMbuild----
CMcall----
CLKbuild----
(None, 2048)
CMcall----


2026-07-14 17:02:10.232109: I external/local_xla/xla/service/service.cc:163] XLA service 0x7d32cc0029f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-07-14 17:02:10.232152: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 Laptop GPU, Compute Capability 8.6
2026-07-14 17:02:10.272318: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-07-14 17:02:10.357594: W tensorflow/compiler/tf2xla/kernels/random_ops.cc:62] Warning: Using tf.random.uniform with XLA compilation will ignore seeds; consider using tf.random.stateless_uniform instead if reproducible behavior is desired. cmgal2_1/genetic__algorithm_1/random_uniform_3/RandomUniform
2026-07-14 17:02:10.361540: W tensorflow/compiler/tf2xla/kernels/random_ops.cc:108] Warning: Using tf.random.uniform with XLA compilation will ignore see

27/28 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 5.8529

2026-07-14 17:02:25.101329: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_393', 4 bytes spill stores, 4 bytes spill loads

2026-07-14 17:02:25.929145: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_650', 4 bytes spill stores, 4 bytes spill loads

2026-07-14 17:02:26.256317: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_393', 52 bytes spill stores, 52 bytes spill loads

2026-07-14 17:02:26.456558: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_650', 4 bytes spill stores, 4 bytes spill loads

2026-07-14 17:02:26.899415: I external/local_xla/x

28/28 ━━━━━━━━━━━━━━━━━━━━ 25s 372ms/step - loss: 5.0575
Epoch 2/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 4.8326
Epoch 3/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 2.8104
Epoch 4/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 2.3024
Epoch 5/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 2.0568
Epoch 6/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 2.6933
Epoch 7/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 2.2954
Epoch 8/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 1.6152
Epoch 9/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 2.0615
Epoch 10/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 2.9606
Epoch 11/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 1.9666
Epoch 12/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 1.0454
Epoch 13/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 0.9225
Epoch 14/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 1.2619
Epoch 15/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 1.2929
Epoch 16/50
2

2026-07-14 17:03:07.563618: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_8', 12 bytes spill stores, 12 bytes spill loads

2026-07-14 17:03:07.977872: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_8', 60 bytes spill stores, 60 bytes spill loads



25/28 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 4.1810

2026-07-14 17:03:13.241978: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_8', 12 bytes spill stores, 12 bytes spill loads

2026-07-14 17:03:13.643343: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_8', 60 bytes spill stores, 60 bytes spill loads



28/28 ━━━━━━━━━━━━━━━━━━━━ 13s 242ms/step - loss: 3.0109
Epoch 2/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 4.0264
Epoch 3/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 2.6339
Epoch 4/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 1.9531
Epoch 5/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 2.6821
Epoch 6/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.7605
Epoch 7/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.6841
Epoch 8/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 2.1050
Epoch 9/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.3596
Epoch 10/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.2316
Epoch 11/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.2487
Epoch 12/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 1.1862
Epoch 13/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.2582
Epoch 14/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.2197
Epoch 15/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.3985
Epoch 16/50
28/28 ━━━━━━

ValueError: Exception encountered when calling Genetic_Algorithm.call().

[1mDimensions must be equal, but are 512 and 1024 for '{{node cmgal4_1/genetic__algorithm_2_1/mul}} = Mul[T=DT_FLOAT](cmgal4_1/genetic__algorithm_2_1/dense_10_1/LeakyRelu, cmgal4_1/genetic__algorithm_2_1/ExpandDims_1)' with input shapes: [1,669,512], [20,1,1024].[0m

Arguments received by Genetic_Algorithm.call():
  • inputs=tf.Tensor(shape=(None, 1024), dtype=float32)
  • model=['<Dense name=dense_8, built=True>', '<Dense name=dense_9, built=True>', '<Dense name=dense_10, built=True>', "'GA'", '<Dense name=dense_11, built=False>']